# Requirement

Read data form flight_time and transformation as below
- 1. Rename fl_date to dep_date
- 2. Compute arr_date
- 3.  Following fields to represent full timestamp
    - 3.1 crs_dep_time
    - 3.2 dep_time
    - 3.3 crs_arr_time
    - 3.4 arr_time

In [0]:
%sql
select * from dev.spark_db.flight_time limit 3

In [0]:
flight_time_df = spark.table("dev.spark_db.flight_time")
flight_time_df.limit(3).display()

In [0]:
flight_time_df_1 = (
    flight_time_df.selectExpr(
        "FL_DATE as DEP_DATE",
        "to_date(DEP_DATE + DEP_TIME + WHEELS_ON + TAXI_IN) AS ARR_DATE",
        "DEP_DATE + CRS_DEP_TIME AS CRS_DEP_TIME",
        "DEP_DATE + DEP_TIME AS DEP_TIME",
        "ARR_DATE + CRS_ARR_TIME AS CRS_ARR_TIME",
        "ARR_DATE + ARR_TIME AS ARR_TIME",
    )
)

flight_time_df_1.limit(3).display()

#selectExpr - we only get those columns which we select in the selectExpr.

2. Can we do it using withColumn or withColumns?

In [0]:
from pyspark.sql.functions import expr

flight_time_df_2 = (
    flight_time_df
        .withColumnRenamed("FL_DATE" , "DEP_DATE")
        .withColumn(
            "ARR_DATE" , expr("to_date(DEP_DATE + DEP_TIME + WHEELS_ON + TAXI_IN)"))
        .withColumns({
            "CRS_DEP_TIME" : expr("DEP_DATE + CRS_DEP_TIME"),
            "DEP_TIME" : expr("DEP_DATE + DEP_TIME"),
            "CRS_ARR_TIME" : expr("ARR_DATE + CRS_ARR_TIME"),
            "ARR_TIME" : expr("ARR_DATE + ARR_TIME")
        })
)

flight_time_df_2.limit(3).display()

3. Alternative approach to write expressions
    Why to use it?
        It gives access to columns functions.

In [0]:
from pyspark.sql.functions import to_date, col

flight_time_df_3 = (
    flight_time_df
        .withColumnRenamed("FL_DATE" , "DEP_DATE")
        .withColumn(
            "ARR_DATE" , to_date(col("DEP_DATE") + col("DEP_TIME") + col("WHEELS_ON") + col("TAXI_IN")))
        .withColumns({
            "CRS_DEP_TIME" : col("DEP_DATE") + col("CRS_DEP_TIME"),
            "DEP_TIME" : col("DEP_DATE") + col("DEP_TIME"),
            "CRS_ARR_TIME" : col("ARR_DATE") + col("CRS_ARR_TIME"),
            "ARR_TIME" : col("ARR_DATE") + col("ARR_TIME")
        })
)

flight_time_df_3.limit(3).display()

In [0]:
flight_time_df_3.where((col("OP_CARRIER_FL_NUM") == 1451) & (col("DEP_DATE") == ' 2000-01-01')).display()

In [0]:
# Using SQL string syntax inside where()
flight_time_df_3.where("OP_CARRIER_FL_NUM = 1451 AND DEP_DATE = '2000-01-01'").display()

In [0]:
# Using SQL string syntax inside where()
flight_time_df_3.where(expr("OP_CARRIER_FL_NUM = 1451 AND DEP_DATE = '2000-01-01'")).display()